# NequIP：基于等变图神经网络的材料性质预测模型（MindSpore 实现）

## 模型简介

**NequIP（Neural Equivariant Interatomic Potential）** 是一种基于 **E(3)-等变图神经网络（E(3)-Equivariant GNN）** 的分子与材料建模方法。  
该模型通过在原子间建立图结构，并在消息传递过程中保持旋转、平移与反射对称性，使得预测结果在三维空间变换下保持一致性。  
相比传统的分子势能面模型，NequIP 具备更高的数据效率与物理一致性，能够在少量样本下学习复杂的原子间相互作用。

## 模型架构

NequIP 的核心由以下部分组成：

- **原子特征嵌入（Embedding Layer）**：将原子种类映射到连续特征空间；
- **等变消息传递层（Equivariant Message Passing）**：基于径向函数与球谐基展开，实现 E(3) 等变的特征更新；
- **能量预测头（Energy/Force Head）**：对每个原子的能量贡献进行汇总，计算体系总能量

## 应用场景

NequIP 模型适用于多种原子尺度建模任务，包括：

- 分子或晶体体系的**能量预测**；
- **分子动力学模拟** 中的力场预测；
- **材料性质预测**，如结合能、晶格常数、带隙估计等。

In [ ]:
import logging
import math
import os
import mindspore as ms
from mindspore import Profiler
from mindspore import nn
from src.dataset import create_training_dataset, _unpack


In [ ]:
import yaml
# 读取配置文件
with open("./rmd.yaml", "r", encoding="utf-8") as f:
    configs = yaml.safe_load(f)

## 数据集说明：`rmd17_uracil.npz`

### 基本信息

- **数据集名称**：`rmd17_uracil.npz`
- **数据来源**：[Revised MD-17 (RMD-17)](https://figshare.com/articles/dataset/Revised_MD17_dataset_rMD17_/12672038)
- **目标分子**：**尿嘧啶（Uracil）**
- **数据格式**：`.npz`（NumPy 压缩格式）
- **原子数**：每构型 **24 个原子**（含氢）

---

### 数据内容概览

该数据集包含尿嘧啶分子在分子动力学（MD）模拟中的 **原子构型轨迹** 与对应的 **高精度量子力学能量和力**，数据经过修正以解决原始 MD-17 的能量不一致性问题。  

所有构型共享相同的原子序数列表（`nuclear_charges`），适用于：

- 分子势能面建模
- 机器学习力场（MLFF）训练
- 图神经网络（GNN）在分子系统上的能量与力联合预测

---

### 原始字段说明（`.npz` 文件）

| 字段名 | 形状 | 类型 | 说明 |
|-------|------|------|------|
| `nuclear_charges` | `(24,)` | int | 每个原子的原子序数（Z），全局共享，不随构型变化 |
| `coords` | `(N, 24, 3)` | float | 所有构型的原子坐标（单位：Å） |
| `energies` | `(N,)` | float | 每个构型的标量能量（单位：kcal/mol） |
| `forces` | `(N, 24, 3)` | float | 每个原子在每个构型中所受的力（单位：kcal/mol/Å） |

---

### 模型输入输出结构

#### 输入字段

| 字段 | 形状 | 说明 |
|------|------|------|
| `x` | `(B * 24,)` | 原子类型索引（int），由 `nuclear_charges` 映射而来，用于嵌入层 |
| `pos` | `(B * 24, 3)` | 展平后的原子坐标 |
| `edge_index` | `(2, num_edges)` | 通过 `radius_graph_full` 构建的边索引（由 `pos` 动态生成） |
| `batch` | `(B * 24,)` | 批次索引，用于区分不同构型 |

#### 输出目标（`label` → `energy`）

| 字段 | 形状 | 说明 |
|------|------|------|
| `energy` | `(B, 1)` | 归一化后的能量标签 |

---

### 自定义数据集

如果想自定义数据集，需要保证包含以下字段：

```json
{
  "nuclear_charges": [6, 7, 8, 1, ...],
  "coords": [[[x, y, z], ...], ...],
  "energies": [...]
}

In [ ]:
from src.dataset import create_training_dataset

data_params = configs.get('data')
trainset, train_edge_index, train_batch, evalset, eval_edge_index, eval_batch, num_type = create_training_dataset(
    config=data_params,
    dtype=ms.float32,
    pred_force=configs.get('pred_force')
)

print("数据集加载完成")
print(f"训练样本数: {data_params.get('n_train')}, 验证样本数: {data_params.get('n_val')}")


NequIP 模型训练流程

In [ ]:
import math
import logging
from mindspore import nn, Profiler
from src.dataset import _unpack
from mindchemistry.cell import Nequip
from tqdm.notebook import tqdm

def generate_learning_rate(learning_rate, warmup_steps, step_num):
    warmup_scale = warmup_steps ** -1.5
    lr = []
    for s in range(1, step_num + 1):
        lr1 = s ** -0.5
        lr2 = s * warmup_scale
        lr.append(learning_rate * min(lr1, lr2))
    return lr

def train(dtype=ms.float32, configs=None):
    """NequIP 模型训练流程"""
    data_params = configs.get('data')
    model_params = configs.get('model')
    optimizer_params = configs.get('optimizer')
    pred_force = configs.get('pred_force')
    is_profiling = configs.get('profiling')
    enable_mix_precision = configs.get('enable_mix_precision')
    ncon_dtype = ms.float16 if enable_mix_precision else ms.float32

    load_ckpt = configs.get('load_ckpt')
    load_ckpt_path = configs.get('load_ckpt_path')
    save_ckpt = configs.get('save_ckpt')
    save_ckpt_interval = configs.get('save_ckpt_interval')
    save_ckpt_path = configs.get('save_ckpt_path')
    if save_ckpt:
        os.makedirs(save_ckpt_path, exist_ok=True)

    logging.info('Loading data...')
    trainset, train_edge_index, train_batch, evalset, eval_edge_index, eval_batch, num_type = create_training_dataset(
        config=data_params, dtype=dtype, pred_force=configs.get('pred_force'))

    # ===== 模型初始化 =====
    logging.info('Initializing model...')
    net = Nequip(
        irreps_embedding_out=model_params.get('irreps_embedding_out'),
        irreps_conv_out=model_params.get('irreps_conv_out'),
        chemical_embedding_irreps_out=model_params.get('chemical_embedding_irreps_out'),
        num_layers=model_params.get('num_layers'),
        num_type=num_type,
        r_max=model_params.get('r_max'),
        hidden_mul=model_params.get('hidden_mul'),
        pred_force=pred_force,
        dtype=dtype,
        ncon_dtype=ncon_dtype
    )

    if load_ckpt:
        logging.info('Loading checkpoint: %s', load_ckpt_path)
        ms.load_checkpoint(load_ckpt_path, net)

    loss_fn = nn.MSELoss()
    metric_fn = nn.MAELoss()
    total_steps_num = optimizer_params.get('num_epoch') * math.ceil(
        data_params.get('n_train') / data_params.get('batch_size'))
    lr_schedule = generate_learning_rate(
        optimizer_params.get('learning_rate'),
        optimizer_params.get('warmup_steps'),
        total_steps_num
    )
    optimizer = nn.Adam(net.trainable_params(), learning_rate=lr_schedule,
                        use_amsgrad=optimizer_params.get('use_amsgrad'))

    # === 前向计算 ===
    def forward(batch, x, pos, edge_src, edge_dst, energy, force, batch_size, sep):
        pred = net(batch, x, pos, edge_src, edge_dst, batch_size)
        if pred_force:
            loss_energy = loss_fn(pred[0], energy)
            loss_force = loss_fn(pred[1], force)
            return loss_energy + 1000. * loss_force
        return loss_fn(pred, energy)

    backward = ms.value_and_grad(forward, None, optimizer.parameters)


    def train_step(batch, x, pos, edge_src, edge_dst, energy, force, size, sep):
        loss_, grads_ = backward(batch, x, pos, edge_src, edge_dst, energy, force, size, sep)
        optimizer(grads_)
        return loss_

    # === 验证 ===
    def validation(eval_batch, eval_edge_index, evalset):
        dataset_size = evalset.get_dataset_size()
        total_eval, total_metric = 0, 0
        for _, data_dict_val in enumerate(evalset.create_dict_iterator()):
            batch_size_val = evalset.get_batch_size()
            inputs_val, label_val = _unpack(data_dict_val)
            pred = net(eval_batch, inputs_val[0], inputs_val[1],
                       eval_edge_index[0], eval_edge_index[1], batch_size_val)
            loss_val = loss_fn(pred, label_val[0]).asnumpy()
            metric = metric_fn(pred, label_val[0]).asnumpy()
            total_eval += loss_val
            total_metric += metric
        loss = total_eval / dataset_size
        metric_mean = total_metric / dataset_size
        logging.info('Eval loss: %8.8f   metric: %.8f', loss, metric_mean)
        return loss, metric_mean

    # === 开始训练 ===
    if is_profiling:
        profiler = Profiler(output_path="profiler_data")

    loss_train, loss_eval, metric = [], [], []
    eval_steps = optimizer_params.get('eval_steps')

    for epoch in range(optimizer_params.get('num_epoch')):
        epoch_loss = 0
        pbar = tqdm(
        total=trainset.get_dataset_size(),
        desc=f"Epoch {epoch+1}/{optimizer_params.get('num_epoch')}",
        leave=True  # 不覆盖
        )
        for _, data_dict in enumerate(trainset.create_dict_iterator()):
            inputs, label = _unpack(data_dict)
            batch_size_train = trainset.get_batch_size()
            loss = train_step(train_batch, inputs[0], inputs[1],
                              train_edge_index[0], train_edge_index[1],
                              label[0], label[1], batch_size_train, False)
            epoch_loss += loss.asnumpy()
            pbar.update(1)
        pbar.close()
        epoch_loss = epoch_loss / trainset.get_dataset_size()
        loss_train.append(epoch_loss)

        if (epoch + 1) % eval_steps == 0:
            loss_val, metric_val = validation(eval_batch, eval_edge_index, evalset)
            loss_eval.append(loss_val)
            metric.append(metric_val)

        if save_ckpt and (epoch + 1) % save_ckpt_interval == 0:
            ms.save_checkpoint(net, f"{save_ckpt_path}/NequIP_rmd_{epoch}.ckpt")

    if is_profiling:
        profiler.analyse()
    if save_ckpt:
        ms.save_checkpoint(net, f"{save_ckpt_path}/NequIP_rmd.ckpt")

    print(f"训练完成，共 {optimizer_params.get('num_epoch')} 个 epoch")
    return loss_train, loss_eval, metric, lr_schedule

In [ ]:
loss_train, loss_eval, metric, lr_schedule = train(dtype=ms.float32, configs=configs)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(loss_train, label="Train Loss")
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

predict定义

In [ ]:
import traceback
import numpy as np


def evaluation_safe(dtype, configs):
    """
    Robust evaluation wrapper for notebook use.
    Returns: (pred_list, loss_mean, metric_mean) or (None, None, None) on error.
    Prints diagnostic info and full traceback on exceptions.
    """
    try:
        # ---- basic checks ----
        if configs is None:
            print("[ERROR] configs is None")
            return None, None, None
        data_params = configs.get('data', {})
        model_params = configs.get('model', {})
        pred_force = configs.get('pred_force', False)
        load_ckpt_path = configs.get('load_ckpt_path')
        save_path = data_params.get('save_path', "./pred_results")
        os.makedirs(save_path, exist_ok=True)

        print("[INFO] dtype:", dtype)
        print("[INFO] load_ckpt_path:", load_ckpt_path)
        if load_ckpt_path is None or not os.path.exists(load_ckpt_path):
            print(f"[ERROR] checkpoint not found at: {load_ckpt_path}")
            print("-> please set configs['load_ckpt_path'] to a valid .ckpt file path")
            return None, None, None

        # ---- load evaluation dataset ----
        print("[INFO] Creating evaluation dataset...")
        _, _, _, evalset, eval_edge_index, eval_batch, num_type = create_training_dataset(
            config=data_params, dtype=dtype, pred_force=pred_force)

        print("[INFO] Eval dataset size:", evalset.get_dataset_size())
        print("[INFO] Eval batch size:", evalset.get_batch_size())
        print("[INFO] num_type:", num_type)

        # ---- build model ----
        print("[INFO] Initializing model...")
        net = Nequip(
            irreps_embedding_out=model_params.get('irreps_embedding_out'),
            irreps_conv_out=model_params.get('irreps_conv_out'),
            chemical_embedding_irreps_out=model_params.get('chemical_embedding_irreps_out'),
            num_layers=model_params.get('num_layers'),
            num_type=num_type,
            r_max=model_params.get('r_max'),
            hidden_mul=model_params.get('hidden_mul'),
            pred_force=pred_force,
            dtype=dtype,
        )

        # ---- load checkpoint ----
        try:
            print(f"[INFO] Loading checkpoint from {load_ckpt_path} ...")
            ms.load_checkpoint(load_ckpt_path, net)
            print("[INFO] Checkpoint loaded successfully.")
        except Exception as e_ckpt:
            print("[ERROR] Failed to load checkpoint:",e_ckpt)
            traceback.print_exc()
            return None, None, None

        loss_fn = nn.MSELoss()
        metric_fn = nn.MAELoss()

        # ---- evaluation loop ----
        dataset_size = evalset.get_dataset_size()
        print("[INFO] Starting evaluation loop...")
        total_loss = total_metric = 0.0
        total_loss_energy = total_loss_force = 0.0
        total_metric_energy = total_metric_force = 0.0

        pred_list = []
        pred_energy_list = []
        pred_force_list=[]
        true_energy_list, true_list = [], []

        for idx, data_dict_val in enumerate(evalset.create_dict_iterator()):
            try:
                batch_size_val = evalset.get_batch_size()
                inputs_val, label_val = _unpack(data_dict_val)

                # Forward (ensure shapes / types)
                pred = net(eval_batch, inputs_val[0], inputs_val[1],
                           eval_edge_index[0], eval_edge_index[1], batch_size_val)

                if not pred_force:
                    loss = loss_fn(pred, label_val[0]).asnumpy()
                    metric = metric_fn(pred, label_val[0]).asnumpy()
                    total_loss += float(loss)
                    total_metric += float(metric)
                    pred_list.append(pred.asnumpy())
                    true_list.append(label_val[0].asnumpy())
                else:
                    loss_energy = float(loss_fn(pred[0], label_val[0]).asnumpy())
                    loss_force = float(loss_fn(pred[1], label_val[1]).asnumpy())
                    metric_energy = float(metric_fn(pred[0], label_val[0]).asnumpy())
                    metric_force = float(metric_fn(pred[1], label_val[1]).asnumpy())
                    total_loss_energy += loss_energy
                    total_loss_force += loss_force
                    total_metric_energy += metric_energy
                    total_metric_force += metric_force
                    pred_energy_list.append(pred[0].asnumpy())
                    pred_force_list.append(pred[1].asnumpy())
                    true_energy_list.append(label_val[0].asnumpy())
            except Exception as e_batch:
                print(f"[ERROR] Exception during eval loop at batch {idx}:",e_batch)
                traceback.print_exc()
                # continue to next batch (or return depending on your preference)
                return None, None, None
        print("pred_list: ", pred_list)
        # ---- summarize and save ----
        if not pred_force:
            loss_mean = total_loss / dataset_size
            metric_mean = total_metric / dataset_size
            np.save(os.path.join(save_path, configs["data"]["pred_file"]), pred_list)
            np.save(os.path.join(save_path, configs["data"]["true_file"]), true_list)
            print("[INFO] Predictions and true values saved to:", save_path)
            print(f"[RESULT] loss_mean: {loss_mean:.8f}  metric_mean: {metric_mean:.8f}")
            return pred_list, loss_mean, metric_mean
        loss_mean = (total_loss_energy / dataset_size, total_loss_force / dataset_size)
        metric_mean = (total_metric_energy / dataset_size, total_metric_force / dataset_size)
        np.save(os.path.join(save_path, configs["data"]["pred_energy_file"]), pred_energy_list)
        np.save(os.path.join(save_path, configs["data"]["true_energy_file"]), true_energy_list)
        print(f"[RESULT] loss_energy_mean: {loss_mean[0]:.8f}  loss_force_mean: {loss_mean[1]:.8f}")
        print(f"[RESULT] metric_energy_mean: {metric_mean[0]:.8f}  metric_force_mean: {metric_mean[1]:.8f}")
        return [pred_energy_list, pred_force_list], loss_mean, metric_mean

    except Exception as e:
        print("[FATAL] Unexpected exception in evaluation_safe:",e)
        traceback.print_exc()
        return None, None, None


In [ ]:
pred_list, loss_mean, metric_mean = evaluation_safe(ms.float32, configs)
print("Returned:", pred_list is not None, loss_mean, metric_mean)

In [ ]:
# 训练epoch达到200个epoch之后，下图的绘制才会有显著效果，这里采用较小的epoch是为了快速验证
import matplotlib.pyplot as plt
pred_energy = np.load("./results/pred.npy", allow_pickle=True)
true_energy = np.load("./results/true.npy", allow_pickle=True)
# 绘制能量对比图
plt.figure(figsize=(10, 5))
num_batches_to_show = min(5, len(pred_energy))  # 只展示前5个batch

for i in range(num_batches_to_show):
    plt.plot(true_energy[i][:50], label=f"True Batch {i}")
    plt.plot(pred_energy[i][:50], "--", label=f"Pred Batch {i}")

plt.title("Energy Prediction Comparison (First Few Batches)")
plt.xlabel("Sample Index in Batch")
plt.ylabel("Energy")
plt.legend(ncol=2, fontsize=8)
plt.show()

In [ ]:

data_path = configs['data'].get("path",None) # 数据集路径
print(configs)
print(data_path)
data = np.load(data_path, allow_pickle=True)

# 取前3个样本
num_samples_to_plot = 3
coords = data['coords']
forces = data['forces']
energies = data['energies']

# 计算整体的力大小范围，用于统一颜色映射
all_force_mags = np.linalg.norm(forces[:, :, :2], axis=2)
vmin, vmax = all_force_mags.min(), all_force_mags.max()
norm = plt.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.Reds

# 创建3个子图并排
fig, axes = plt.subplots(1, num_samples_to_plot, figsize=(15, 5), constrained_layout=True)

for i, ax in enumerate(axes):
    xy_coords = coords[i][:, :2]
    force_xy = forces[i][:, :2]
    force_mags = np.linalg.norm(force_xy, axis=1)
    colors = cmap(norm(force_mags))

    # 原子点
    ax.scatter(xy_coords[:, 0], xy_coords[:, 1], s=100, c='skyblue', edgecolors='k', label='Atoms')

    # 力箭头（带颜色）
    scale = 0.01
    for j in range(xy_coords.shape[0]):
        ax.arrow(xy_coords[j, 0], xy_coords[j, 1],
                 force_xy[j, 0] * scale, force_xy[j, 1] * scale,
                 color=colors[j], width=0.004, head_width=0.05, alpha=0.8)

    # 原子编号
    for j, (x, y) in enumerate(xy_coords):
        ax.text(x + 0.02, y + 0.02, str(j), fontsize=8)

    ax.set_aspect('equal')
    ax.set_title(f"Sample {i}\nEnergy: {energies[i]:.2f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

# 添加颜色条（力大小）
cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label("Force Magnitude (a.u.)")

plt.show()